# Prompt Engineering — Baseline

Prompt engineering is **iterative refinement**: write a prompt, *evaluate* it, apply a technique, *re-evaluate*, repeat. Each iteration should move the eval score measurably.

```
1. Set a goal
2. Write an initial prompt
3. Evaluate it            ┐
4. Apply a technique       ├─ repeat 3–4 until satisfied
5. Re-evaluate            ┘
```

**Running example for the whole section:** a prompt that generates a **one-day meal plan for an athlete** from their height, weight, goal, and dietary restrictions.

This lesson establishes the **baseline** with a deliberately naive prompt. Don't be discouraged by a low score — ~2/10 is typical for a first attempt. The point is to have a number to beat.

> **Two structural choices for this section:**
> 1. **Shared harness.** The course's `PromptEvaluator` (dataset generation + concurrent model grading + HTML report) is ~400 lines. Instead of pasting it into all six lessons, it lives in **`prompt_evaluator.py`** in this folder and each notebook imports it. (Runs on **Haiku 4.5** — the harness uses both assistant prefilling *and* `temperature`, which each 400 on flagship models.)
> 2. **One dataset, reused.** We generate the test dataset **once here** and every technique lesson loads *this same* `dataset.json`. Holding the dataset constant means score changes are attributable to the **prompt**, not to different test cases.

## Load the shared harness

We add this section folder to `sys.path` (derived from the repo root, so it works regardless of the kernel's cwd) and import the evaluator + helpers.

In [1]:
import sys
import os
from dotenv import find_dotenv

REPO_ROOT = os.path.dirname(find_dotenv())
SECTION = os.path.join(REPO_ROOT, "building-with-the-claude-api", "03-prompt-engineering")
if SECTION not in sys.path:
    sys.path.insert(0, SECTION)

from prompt_evaluator import PromptEvaluator, add_user_message, chat

DATASET = os.path.join(SECTION, "dataset.json")
print("Harness loaded. Dataset ->", DATASET)

Harness loaded. Dataset -> c:\Development\anthropic-cert\building-with-the-claude-api\03-prompt-engineering\dataset.json


## Create the evaluator

`max_concurrent_tasks` controls parallelism. Start low (1–3) to avoid rate-limit errors; raise it if your quota allows.

In [2]:
evaluator = PromptEvaluator(max_concurrent_tasks=3)

## Generate the dataset (once for the whole section)

Describe the task and the inputs the prompt needs; the evaluator invents diverse athlete scenarios and matching solution criteria. Keep `num_cases` low (2–3) for fast iteration. **The later technique lessons reuse this exact file** — rerun this cell only if you want a fresh dataset (which resets the baseline for comparison).

In [3]:
dataset = evaluator.generate_dataset(
    task_description="Write a compact, concise 1 day meal plan for a single athlete",
    prompt_inputs_spec={
        "height": "Athlete's height in cm",
        "weight": "Athlete's weight in kg",
        "goal": "Goal of the athlete",
        "restrictions": "Dietary restrictions of the athlete",
    },
    output_file=DATASET,
    num_cases=3,
)

dataset

Generated 1/3 test cases
Generated 2/3 test cases
Generated 3/3 test cases


[{'prompt_inputs': {'height': '178 cm',
   'weight': '68 kg',
   'goal': 'Marathon training day with 3000+ calorie intake and carbohydrate loading for endurance performance',
   'restrictions': 'Vegetarian, no nuts'},
  'solution_criteria': ['Total daily calories meet or exceed 3000 calories',
   'Carbohydrates comprise at least 60% of total calories',
   'Includes hydration timing recommendations aligned with training schedule',
   'All meals and snacks respect vegetarian and nut-free restrictions'],
  'task_description': 'Write a compact, concise 1 day meal plan for a single athlete',
  'scenario': 'Testing with a high-endurance athlete (marathon runner) requiring 3000+ calories with emphasis on carbohydrate loading and hydration timing'},
 {'prompt_inputs': {'height': '178 cm',
   'weight': '72 kg',
   'goal': 'Build muscle and improve endurance for competitive cycling',
   'restrictions': 'Vegan and ketogenic (low-carb, high-fat plant-based)'},
  'solution_criteria': ['Meal plan in

## The naive baseline prompt

As basic as it gets — no structure, no guidelines, no example. This is what we'll improve, one technique at a time.

In [4]:
def run_prompt(prompt_inputs):
    prompt = f"""
What should this person eat?

- Height: {prompt_inputs["height"]}
- Weight: {prompt_inputs["weight"]}
- Goal: {prompt_inputs["goal"]}
- Dietary restrictions: {prompt_inputs["restrictions"]}
"""

    messages = []
    add_user_message(messages, prompt)
    return chat(messages)

## Run the baseline evaluation

`extra_criteria` are treated as **mandatory** by the grader — violating them caps the score at 3. This is what makes the naive prompt score so low: it rarely produces caloric totals, macro breakdowns, and exact portions/timing unprompted. Outputs are written to a JSON blob and an HTML report you can open in a browser.

In [5]:
results = evaluator.run_evaluation(
    run_prompt_function=run_prompt,
    dataset_file=DATASET,
    extra_criteria="""
The output should include:
- Daily caloric total
- Macronutrient breakdown
- Meals with exact foods, portions, and timing
""",
    json_output_file=os.path.join(SECTION, "output-01-intro.json"),
    html_output_file=os.path.join(SECTION, "output-01-intro.html"),
)

Graded 1/3 test cases
Graded 2/3 test cases
Graded 3/3 test cases
Average score: 2.3333333333333335


## Baseline established

Open **`output-01-intro.html`** to see the per-case scores and the grader's reasoning — that reasoning is your roadmap for what to fix. A low number here (often ~2–3/10) is expected and *good*: it leaves plenty of room to demonstrate that each technique actually helps.

Next lessons apply techniques one at a time — **being clear and direct**, **being specific**, **structure with XML tags**, **providing examples** — re-running against this same dataset so we can watch the score climb.